# Photon-counting CT and material decomposition

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/photon_counting_ct)

A photon-counting detector sorts every X-ray into energy bins, so each ray gives several Beer–Lambert measurements $\lambda_{rb} = N_0 R_{be}S_e e^{-\mu_{em}A_{rm}}$ instead of one. A batched maximum-likelihood fit (autograd gradient, Fisher-scoring steps) turns the Poisson counts of every ray into water, bone and iodine line integrals; filtered back-projection makes material maps and virtual monoenergetic images, and separates iodine from bone that a conventional scanner shows as identical.

**Vector form**

$$\lambda_b = N_0 \int R_b(E)\,S(E)\,\exp\!\Big(-\sum_m \mu_m(E)\int_{\text{ray}} a_m(\mathbf r)\,dl\Big)\,dE \qquad \hat{\mathbf A} = \arg\min_{\mathbf A} \sum_b \big(\lambda_b(\mathbf A) - y_b\ln\lambda_b(\mathbf A)\big)$$

**Index (tensor) form**

$$A_{rm} = \textstyle\int a_m\,dl \qquad \lambda_{rb} = N_0\,R_{be}\,S_e\,e^{-\mu_{em}A_{rm}} \qquad F_{rmn} = J_{rbm}J_{rbn}/\lambda_{rb} \qquad A_{rm} \leftarrow A_{rm} - (F^{-1})_{rmn}\,\partial_{A_{rn}}\mathcal L \qquad \mu_{yx}(E) = \mu_m(E)\,a_{myx}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a_{myx}$ | `phantom(x, bone_fraction)` | (M, N, N) | basis amounts per pixel: water fraction, bone fraction, iodine in mg/mL |
| $A_{rm}$ | `basis_sinograms(maps, theta, x)` | (A, S, M) | Radon transform of each basis map: cm of water, cm of bone, mg/cm² of iodine along ray $r = (\theta, s)$ |
| $\mu_{em}$ | `basis_attenuation(e)` | (E, M) | attenuation of each basis material on a 1 keV grid (XCOM, log-log interpolated, iodine K-edge at 33.2 keV) |
| $S_e,\ R_{be}$ | `spectrum(kvp, e), bin_response(e, T, sigma)` | (E,), (B, E) | filtered tube spectrum; probability that a photon of energy $E_e$ is counted in bin $b$ (Gaussian energy resolution) |
| $\lambda_{rb}$ | `n0 * einsum('be,e,re->rb', R, s, exp(-einsum('em,rm->re', mu, A)))` | (R, B) | Beer–Lambert per bin: repeated $m$ then $e$ ⇒ summed; the counts are $y_{rb} \sim \mathrm{Poisson}(\lambda_{rb})$ |
| $\partial_{A_{rn}}\mathcal L$ | `autograd.grad(poisson_nll(y, lam).sum(), A)` | (R, M) | gradient of the Poisson negative log-likelihood; rays are independent, so one backward pass gives every ray its own gradient |
| $F_{rmn}$ | `einsum('rbm,rbn,rb->rmn', J, J, 1 / lam)` | (R, M, M) | Fisher information, repeated $b$ ⇒ summed; $J_{rbm} = \partial\lambda_{rb}/\partial A_{rm}$ |
| $(F^{-1})_{rmn}\,\partial_{A_{rn}}\mathcal L$ | `linalg.solve(F, grad)` | (R, M) | one $3\times 3$ solve per ray, batched: the Fisher-scoring (Newton-like) step |
| $\mu_{yx}(E)$ | `einsum('...m,myx->...yx', mu_E, maps)` | (K, N, N) | virtual monoenergetic image at energy $E$ from the reconstructed basis maps, shown in HU |

Photon-counting detectors (cadmium telluride or silicon) register single photons and compare each pulse height with several thresholds, so the count in each energy bin is a separate Beer–Lambert measurement. The first photon-counting CT scanner was cleared by the US FDA in 2021, and since then spectral imaging, sharper resolution and lower electronic noise have moved from research into clinics. Three materials need at least three bins: water and bone differ mainly through the photoelectric effect, and iodine adds its K-edge at 33.2 keV, so a threshold near 35 keV helps (try the 25, 50, 75 keV setting). The bone insert is diluted until it has the same energy-weighted attenuation as the 10 mg/mL iodine insert, so the conventional image shows the two as equal; the material maps tell them apart. Virtual monoenergetic images $\mu_m(E)a_m$ boost iodine contrast at low energy and have no beam-hardening cupping, which the conventional image shows. Three-material decomposition amplifies noise (the basis attenuation curves are nearly parallel), so the iodine error falls only as $1/\sqrt{N_0}$ and is a few mg/mL per pixel at moderate dose; clinical systems often use two basis materials, denoising and iterative reconstruction. Simplifications: parallel beams, ideal detector apart from Gaussian energy resolution (no charge sharing, pile-up or K-escape), no scatter, rounded attenuation tables. Educational model, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/photon_counting_ct/main.py
"""Main loop: parameters → tensors → photon counts → material decomposition → reconstruction → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

THRESHOLDS = {"25,35,50,70": [25.0, 35.0, 50.0, 70.0], "25,50,75": [25.0, 50.0, 75.0]}


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    kvp, n0 = params["kvp"], params["dose"]
    e = tensors.energies(kvp, dev)  # (E,) keV
    mu = tensors.basis_attenuation(e)  # (E, M)
    s = tensors.spectrum(kvp, e)  # (E,)
    R = tensors.bin_response(e, THRESHOLDS[params["thresholds"]], params["sigma_kev"])  # (B, E)
    w_bone = tensors.matched_bone_fraction(mu, s, e, tensors.MATCHED_IODINE)

    n = 64 if fast else params["n_pixels"]
    n_angles = 60 if fast else params["n_angles"]
    x = tensors.grid(n, dev)  # (N,)
    maps, rois, labels = tensors.phantom(x, w_bone)  # (M, N, N), (5, N, N)
    theta = tensors.angles(n_angles, dev)  # (A,)

    # Scan: basis line integrals → expected counts per bin → Poisson counts
    A_true = solve.basis_sinograms(maps, theta, x)  # (A, S, M)
    rays = A_true.reshape(-1, 3)  # (R, M), R = A·S
    lam_unit = equation.expected_counts(1.0, s, R, mu, rays)  # (R, B) counts per open-beam photon
    y = solve.count_photons(lam_unit * n0, seed=0)  # (R, B)
    print(f"{kvp:.0f} kVp, {R.shape[0]} bins, thresholds {params['thresholds']} keV, σ = {params['sigma_kev']} keV")
    print(f"open beam per bin: {', '.join(f'{v:.0f}' for v in (n0 * R @ s).tolist())} photons per ray")
    print(f"thickest ray: {y.min(0).values.int().tolist()} counts per bin (low bin starves first)")
    print(
        f"bone insert: {100 * w_bone:.1f}% cortical bone in water, matched to {tensors.MATCHED_IODINE:g} mg/mL iodine"
    )

    # Decompose every ray (one batched ML fit), then reconstruct each basis sinogram by FBP
    A_hat = solve.decompose(y, n0, s, R, mu).reshape(A_true.shape)  # (A, S, M)
    noise_free = solve.decompose(lam_unit * 1e9, 1e9, s, R, mu, n_iter=8)
    print(f"decomposition of noise-free data: max error {(noise_free - rays).abs().max():.1e}")

    conventional = equation.energy_integrating(n0, s, e, mu, rays).reshape(n_angles, n)  # (A, S) −ln(I/I0)
    sinos = torch.cat([A_hat.permute(2, 0, 1), conventional[None]])  # (4, A, S)
    Q, images = solve.reconstruct(sinos, theta, x)  # (4, A, N, N), (4, N, N): water, bone, iodine, conventional

    print("ROI means      " + "  ".join(f"{lab:>7}" for lab in labels))
    for name, k in (("iodine mg/mL", 2), ("bone %", 1), ("conv μ 1/cm", 3)):
        scale = 100.0 if k == 1 else 1.0
        print(f"{name:13} " + "  ".join(f"{v * scale:7.2f}" for v in solve.roi_means(images[k], rois).tolist()))

    vmi_e = torch.arange(40.0, kvp - 9, 4.0, device=dev, dtype=e.dtype)  # keV energies for the VMI sweep
    vmi_mu = tensors.basis_attenuation(vmi_e)  # (K, M)
    vmi = equation.hounsfield(equation.monoenergetic(vmi_mu, images[:3]), vmi_mu[:, 0, None, None])  # (K, N, N)
    vmi_rois = torch.stack([solve.roi_means(v, rois) for v in vmi])  # (K, 5)
    i40, i70 = 0, int(torch.argmin((vmi_e - 70).abs()))
    print(f"VMI 40 keV: I10 {vmi_rois[i40, 2]:.0f} HU, bone {vmi_rois[i40, 4]:.0f} HU;  ", end="")
    print(f"70 keV: I10 {vmi_rois[i70, 2]:.0f} HU, bone {vmi_rois[i70, 4]:.0f} HU")

    doses = [n0 * f for f in ([0.25, 1.0] if fast else [0.1, 0.3, 1.0, 3.0, 10.0])]
    errors = solve.iodine_error_vs_dose(lam_unit, A_true.shape, s, R, mu, theta, x, maps, rois, doses)
    for d, err in zip(doses, errors):
        print(f"  N0 = {d:9.3g} photons/ray: iodine RMS error {err:.2f} mg/mL")

    detected = R * s  # (B, E) open-beam spectrum seen by each bin
    return plot.animate(
        x, theta, e, detected, mu, Q, images, maps, vmi_e, vmi, vmi_rois, labels, doses, errors,
        out_dir / "photon_counting_ct.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/photon_counting_ct/equation.py
r"""Physics → tensor translation.

Every pixel is a mix of basis materials m (water, bone, iodine) with amounts a_{myx}, so its attenuation at
energy E_e is μ_{eyx} = μ_{em} a_{myx}. Along a ray r (one angle θ, one detector bin s) only the basis line
integrals A_{rm} = ∫ a_m dl matter, and Beer–Lambert for a polychromatic beam counted in energy bin b reads

    λ_{rb} = N_0\, R_{be}\, S_e \exp(-\mu_{em} A_{rm})          (e, m summed)

Indices: r = ray, b = detector energy bin, e = photon energy, m = basis material.
S_e is the tube spectrum (sums to 1), R_{be} the probability that a photon of energy E_e lands in bin b.
The counts are Poisson, y_{rb} ~ Poisson(λ_{rb}), so the decomposition minimises the negative log-likelihood

    \mathcal L(A) = \sum_{rb} \lambda_{rb}(A) - y_{rb} \ln \lambda_{rb}(A)

ray by ray (all rays in one tensor). Its Fisher information, used for the Newton-like steps, is

    F_{rmn} = J_{rbm} J_{rbn} / \lambda_{rb},   J_{rbm} = \partial\lambda_{rb}/\partial A_{rm}   (b summed)

A conventional energy-integrating detector adds up the deposited energy instead:  I_r = N_0 E_e S_e T_{re}.
The Radon transform and filtered back-projection are the same as in radon_ct.
"""

import torch
import torch.nn.functional as F


def line_integrals_mu(mu: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    r"""\int \mu(E_e)\,dl = \mu_{em} A_{rm}.   mu (E, M), A (R, M) → (R, E);  m summed."""
    return torch.einsum("em,rm->re", mu, A)


def expected_counts(n0: float, s: torch.Tensor, R: torch.Tensor, mu: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    r"""Beer–Lambert per bin: λ_{rb} = N_0 R_{be} S_e \exp(-\mu_{em}A_{rm}).   → (R, B);  e summed."""
    T = torch.exp(-line_integrals_mu(mu, A))  # (R, E) transmission at each energy
    return n0 * torch.einsum("be,e,re->rb", R, s, T)


def poisson_nll(y: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    r"""Negative Poisson log-likelihood per ray, \sum_b (λ_{rb} − y_{rb} \ln λ_{rb}) (constant term dropped): (R,)."""
    return (lam - y * torch.log(lam)).sum(-1)


def jacobian(n0: float, s: torch.Tensor, R: torch.Tensor, mu: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    r"""J_{rbm} = ∂λ_{rb}/∂A_{rm} = −N_0 R_{be} S_e T_{re} μ_{em}.   → (R, B, M);  e summed."""
    T = torch.exp(-line_integrals_mu(mu, A))
    return -n0 * torch.einsum("be,e,re,em->rbm", R, s, T, mu)


def fisher(J: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    r"""F_{rmn} = J_{rbm} J_{rbn} / λ_{rb}: expected curvature of the NLL, one M×M matrix per ray.  b summed."""
    return torch.einsum("rbm,rbn,rb->rmn", J, J, 1.0 / lam)


def energy_integrating(n0: float, s: torch.Tensor, e: torch.Tensor, mu: torch.Tensor, A: torch.Tensor):
    r"""Conventional detector, signal ∝ deposited energy: I_r = N_0 E_e S_e T_{re} (e summed) → −ln(I/I_0), (R,)."""
    T = torch.exp(-line_integrals_mu(mu, A))
    signal = n0 * torch.einsum("e,e,re->r", e, s, T)
    return -torch.log(signal / (n0 * (e * s).sum()))


def monoenergetic(mu_at_e: torch.Tensor, maps: torch.Tensor) -> torch.Tensor:
    r"""Virtual monoenergetic image μ_{yx}(E) = μ_m(E) a_{myx}.   mu_at_e (M,) or (K, M), maps (M, Y, X);  m summed."""
    return torch.einsum("...m,myx->...yx", mu_at_e, maps)


def hounsfield(mu_map: torch.Tensor, mu_water: torch.Tensor) -> torch.Tensor:
    """CT number HU = 1000 (μ − μ_water) / μ_water."""
    return 1000.0 * (mu_map - mu_water) / mu_water


# --- Radon transform and filtered back-projection (as in radon_ct), batched over a leading channel axis ---


def radon(f: torch.Tensor, theta: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    r"""p_{c\theta s} = f_{c\theta ts}\,\Delta t  for C images at once.  f (C, N, N) → (C, A, S), t summed."""
    c, sn = torch.cos(theta)[:, None, None], torch.sin(theta)[:, None, None]  # (A, 1, 1)
    x = s[None, None, :] * c - s[None, :, None] * sn  # (A, T, S): sample points of every ray
    y = s[None, None, :] * sn + s[None, :, None] * c
    points = torch.stack([x, y], dim=-1)
    image = f[None].expand(len(theta), *f.shape)  # (A, C, N, N)
    samples = F.grid_sample(image, points, mode="bilinear", align_corners=False)  # (A, C, T, S)
    return torch.einsum("acts->cas", samples) * (s[1] - s[0])


def ramp_filter(n_det: int, ds: float, device=None, dtype=torch.float64) -> tuple[torch.Tensor, int]:
    """Band-limited ramp |ν| × sinc(ν Δs) (Shepp–Logan window) on a zero-padded detector of P ≥ 2S bins."""
    pad = 1 << (2 * n_det - 1).bit_length()
    idx = torch.arange(pad, device=device)
    n = torch.where(idx < pad // 2, idx, idx - pad).to(dtype)
    h = torch.where(n % 2 == 1, -1.0 / (torch.pi * n * ds) ** 2, torch.zeros_like(n))
    h[0] = 1.0 / (4 * ds**2)
    nu = torch.fft.fftfreq(pad, d=ds, device=device).to(dtype)
    return ds * torch.fft.fft(h).real * torch.sinc(nu * ds), pad


def smear_filtered(p: torch.Tensor, theta: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""Ramp-filter each projection and spread it back: Q_{c\theta yx} = q_{c\theta}(x\cos\theta + y\sin\theta).

    p (C, A, S) → (C, A, Y, X). Summing over θ with Δθ = π/A gives the filtered back-projection.
    """
    H, pad = ramp_filter(p.shape[-1], float(x[1] - x[0]), p.device, p.dtype)
    q = torch.fft.ifft(torch.fft.fft(p, n=pad, dim=-1) * H, dim=-1).real[..., : p.shape[-1]]  # (C, A, S)
    s = x[None, None, :] * torch.cos(theta)[:, None, None] + x[None, :, None] * torch.sin(theta)[:, None, None]
    points = torch.stack([s, torch.zeros_like(s)], dim=-1)  # (A, Y, X, 2)
    q_img = q.permute(1, 0, 2)[:, :, None, :]  # (A, C, 1, S): one 1-pixel-high image per angle
    return F.grid_sample(q_img, points, mode="bilinear", align_corners=False).permute(1, 0, 2, 3)


def backproject(Q: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    r"""f_{cyx} = Q_{c\theta yx}\,\Delta\theta  (θ summed).   (C, A, Y, X) → (C, Y, X)"""
    return torch.einsum("cayx,a->cyx", Q, torch.full_like(theta, torch.pi / len(theta)))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/photon_counting_ct/tensors.py
"""Build the concrete tensors: energy grid E_e, basis attenuation μ_{em}, tube spectrum S_e, bin response R_{be},
the phantom's basis-material maps a_{myx} and the projection angles θ.

Mass attenuation coefficients (cm²/g) are rounded NIST XCOM values (the same sources as xray_attenuation),
interpolated log-log onto a 1 keV grid. Iodine has its K-edge at 33.17 keV, so its table lists that energy
twice: the value just below and just above the edge. Good enough to see the physics, not for clinical use.
"""

import math

import torch

MATERIALS = ["water", "bone", "iodine"]  # basis materials m = 0, 1, 2
K_EDGE = 33.169  # keV, iodine K-shell binding energy

# (energy keV, μ/ρ cm²/g) tables
WATER = ([20, 30, 40, 50, 60, 80, 100, 150], [0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707, 0.1505])
BONE = ([20, 30, 40, 50, 60, 80, 100, 150], [4.001, 1.331, 0.6655, 0.4242, 0.3148, 0.2229, 0.1855, 0.1480])
IODINE = (
    [20, 30, K_EDGE, K_EDGE, 40, 50, 60, 80, 100, 120, 150],
    [25.4, 8.65, 6.55, 36.0, 22.1, 12.3, 7.6, 3.5, 1.94, 1.2, 0.69],
)
ALUMINIUM = ([20, 30, 40, 50, 60, 80, 100, 150], [3.441, 1.128, 0.5685, 0.3681, 0.2778, 0.2018, 0.1704, 0.1378])
RHO_WATER, RHO_BONE, RHO_AL = 1.0, 1.92, 2.699  # g/cm³ (ICRU cortical bone)

FOV_HALF_WIDTH = 12.0  # cm: the image grid [-1, 1]² covers a 24 cm field of view
IODINE_INSERTS = [2.5, 5.0, 10.0, 15.0]  # mg/mL iodine dissolved in water
MATCHED_IODINE = 10.0  # the bone insert is diluted until it looks like this iodine insert on conventional CT


def energies(kvp: float, device: torch.device, dtype=torch.float64) -> torch.Tensor:
    """Photon energies E_e = 20, 21, … kVp−1 keV: shape (E,)."""
    return torch.arange(20.0, float(kvp), 1.0, device=device, dtype=dtype)


def loglog(e: torch.Tensor, table: tuple[list, list]) -> torch.Tensor:
    """Interpolate μ/ρ linearly in (log E, log μ). A repeated table energy is an absorption edge:
    searchsorted(right=True) puts energies above it in the interval that starts with the post-edge value."""
    e_tab = torch.tensor(table[0], dtype=e.dtype, device=e.device)
    mu_tab = torch.tensor(table[1], dtype=e.dtype, device=e.device)
    i = torch.searchsorted(e_tab, e, right=True).clamp(1, len(e_tab) - 1)  # table interval [i-1, i]
    lo, hi = torch.log(e_tab[i - 1]), torch.log(e_tab[i])
    w = (torch.log(e) - lo) / (hi - lo)
    return torch.exp((1 - w) * torch.log(mu_tab[i - 1]) + w * torch.log(mu_tab[i]))


def basis_attenuation(e: torch.Tensor) -> torch.Tensor:
    """μ_{em} of shape (E, 3): water and bone in 1/cm per (cm of material), iodine in 1/cm per (mg/cm³).

    A pixel with water fraction a_w, bone fraction a_b and c mg/mL iodine has μ_e = μ_{em} a_m.
    """
    water = loglog(e, WATER) * RHO_WATER
    bone = loglog(e, BONE) * RHO_BONE
    iodine = loglog(e, IODINE) / 1000.0  # (cm²/g)·(mg/cm³)/(1000 mg/g)
    return torch.stack([water, bone, iodine], dim=1)


def spectrum(kvp: float, e: torch.Tensor, filter_mm_al: float = 3.0) -> torch.Tensor:
    """S_e: Kramers' photon-number spectrum (kVp − E)/E behind an aluminium filter; sums to 1. (E,)"""
    kramers = (kvp - e).clamp_min(0.0) / e
    filtered = kramers * torch.exp(-loglog(e, ALUMINIUM) * RHO_AL * filter_mm_al / 10.0)
    return filtered / filtered.sum()


def bin_response(e: torch.Tensor, thresholds: list[float], sigma_kev: float) -> torch.Tensor:
    """R_{be}: probability that a photon of energy E_e is counted in bin b (thresholds T_b ≤ E' < T_{b+1}).

    The detector measures E' = E + Gaussian noise of width σ, so R_{be} = Φ((T_{b+1} − E)/σ) − Φ((T_b − E)/σ),
    with T_{B} = ∞. Photons below the lowest threshold are not counted (that threshold rejects electronic noise).
    """
    edges = torch.tensor([*thresholds, math.inf], dtype=e.dtype, device=e.device)  # (B+1,)
    cdf = 0.5 * (1 + torch.erf((edges[:, None] - e[None, :]) / (math.sqrt(2) * max(sigma_kev, 1e-3))))
    return cdf[1:] - cdf[:-1]  # (B, E)


def matched_bone_fraction(mu: torch.Tensor, s: torch.Tensor, e: torch.Tensor, conc: float) -> float:
    """Bone volume fraction w that an energy-integrating scanner cannot tell from `conc` mg/mL iodine.

    Use the spectrum hardened by 10 cm of water, energy-weighted as the detector weights it, and solve
    w μ̄_bone + (1 − w) μ̄_water = μ̄_water + c μ̄_iodine for w.
    """
    weight = s * e * torch.exp(-mu[:, 0] * 10.0)
    weight = weight / weight.sum()
    mw, mb, mi = (weight @ mu).tolist()
    return conc * mi / (mb - mw)


def grid(n: int, device: torch.device, dtype=torch.float64) -> torch.Tensor:
    """Pixel centres in [-1, 1] (units of the FOV half width): (N,). The detector bins s use the same grid."""
    return (torch.arange(n, device=device, dtype=dtype) + 0.5) * (2.0 / n) - 1.0


def phantom(x: torch.Tensor, bone_fraction: float):
    """A 20 cm water cylinder with five 3.2 cm inserts on a ring: four iodine concentrations and one diluted bone.

    Returns the basis maps a_{myx} (3, N, N): water fraction, bone fraction, iodine mg/mL;
    the insert masks (5, N, N) bool (slightly shrunk, for ROI statistics) and their labels.
    """
    yy, xx = torch.meshgrid(x, x, indexing="ij")
    r = FOV_HALF_WIDTH
    maps = torch.zeros(3, len(x), len(x), dtype=x.dtype, device=x.device)
    body = xx**2 + yy**2 <= (10.0 / r) ** 2
    maps[0][body] = 1.0
    labels = [f"I {c:g}" for c in IODINE_INSERTS] + ["bone"]
    rois = []
    for k in range(5):
        phi = math.pi / 2 + 2 * math.pi * k / 5
        cx, cy = 5.5 / r * math.cos(phi), 5.5 / r * math.sin(phi)
        d2 = (xx - cx) ** 2 + (yy - cy) ** 2
        inside = d2 <= (1.6 / r) ** 2
        if k < 4:
            maps[2][inside] = IODINE_INSERTS[k]  # iodine dissolved in water: water fraction stays 1
        else:
            maps[0][inside] = 1.0 - bone_fraction
            maps[1][inside] = bone_fraction
        rois.append(d2 <= (1.1 / r) ** 2)
    return maps, torch.stack(rois), labels


def angles(n: int, device: torch.device, dtype=torch.float64) -> torch.Tensor:
    """θ_a = π a / A: a half turn of parallel-beam projections. (A,)"""
    return torch.arange(n, device=device, dtype=dtype) * (math.pi / n)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/photon_counting_ct/solve.py
"""Scan the phantom, count photons per bin, decompose every ray into basis line integrals and reconstruct."""

import torch

from problems._common.progress import steps

from . import equation, tensors


def basis_sinograms(maps: torch.Tensor, theta: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Basis line integrals A_{θsm} in physical units (cm of water, cm of bone, mg/cm² iodine): (A, S, M)."""
    return (equation.radon(maps, theta, x) * tensors.FOV_HALF_WIDTH).permute(1, 2, 0)


def count_photons(lam: torch.Tensor, seed: int) -> torch.Tensor:
    """Poisson counts y_{rb} ~ Poisson(λ_{rb}) with a fixed seed."""
    gen = torch.Generator(device=lam.device).manual_seed(seed)
    return torch.poisson(lam, generator=gen)


def decompose(y: torch.Tensor, n0: float, s, R, mu, n_iter: int = 15) -> torch.Tensor:
    """Maximum-likelihood basis line integrals for every ray at once: y (R, B) counts → A (R, M).

    1. Start from weighted least squares on the log counts, −ln(y_b / y_{0b}) ≈ μ̄_{bm} A_m (μ̄ = bin-averaged μ,
       weights ≈ y_b because Var ln y ≈ 1/y). This ignores beam hardening inside a bin, so it is biased.
    2. Fisher scoring: the gradient of the Poisson NLL comes from autograd (rays are independent, so the gradient of
       the summed NLL is every ray's own gradient), the curvature from the Fisher matrix F_{rmn}, and
       A ← A − F⁻¹ ∇L solves one small M×M system per ray, batched.
    """
    w0 = R * s  # (B, E) open-beam photons per bin and energy
    mu_bar = (w0 @ mu) / w0.sum(1, keepdim=True)  # (B, M) bin-averaged attenuation at zero thickness
    log_ratio = -torch.log(y.clamp_min(0.5) / (n0 * w0.sum(1)))  # (R, B)
    W = y.clamp_min(1.0)  # (R, B) inverse variance of the log counts
    normal = torch.einsum("rb,bm,bn->rmn", W, mu_bar, mu_bar)
    rhs = torch.einsum("rb,bm,rb->rm", W, mu_bar, log_ratio)
    A = torch.linalg.solve(normal, rhs)  # (R, M) initial guess

    eye = 1e-12 * torch.eye(mu.shape[1], dtype=A.dtype, device=A.device)
    for _ in range(n_iter):
        A = A.detach().requires_grad_(True)
        lam = equation.expected_counts(n0, s, R, mu, A)
        (grad,) = torch.autograd.grad(equation.poisson_nll(y, lam).sum(), A)  # (R, M)
        with torch.no_grad():
            F = equation.fisher(equation.jacobian(n0, s, R, mu, A), lam)  # (R, M, M)
            step = torch.linalg.solve(F + eye, grad)
            A = A - step.clamp(-5.0, 5.0)  # keep early steps sane far from the optimum
    return A.detach()


def reconstruct(sino: torch.Tensor, theta: torch.Tensor, x: torch.Tensor):
    """Per-angle filtered back-projections and the final images of C sinograms (C, A, S) given in physical units.

    Returns (Q (C, A, N, N), images (C, N, N)); a running sum of Q over θ shows the image forming.
    """
    Q = equation.smear_filtered(sino / tensors.FOV_HALF_WIDTH, theta, x)
    return Q, equation.backproject(Q, theta)


def iodine_error_vs_dose(lam_unit, A_shape, s, R, mu, theta, x, maps, rois, doses, seed=1):
    """For each dose N_0: Poisson counts → decomposition → FBP; RMS error of the iodine map inside the inserts (mg/mL).

    lam_unit is the noise-free count per photon of open beam, λ_{rb}/N_0, of shape (R, B).
    """
    inside = rois.any(0)
    errors = []
    for k in steps(len(doses), "doses"):
        n0 = doses[k]
        y = count_photons(lam_unit * n0, seed + k)
        A = decompose(y, n0, s, R, mu).reshape(*A_shape)
        _, img = reconstruct(A[..., 2:].permute(2, 0, 1), theta, x)
        errors.append(float(((img[0] - maps[2])[inside] ** 2).mean().sqrt()))
    return errors


def roi_means(img: torch.Tensor, rois: torch.Tensor) -> torch.Tensor:
    """Mean of an image inside each ROI: (K,)."""
    return torch.einsum("kyx,yx->k", rois.to(img.dtype), img) / rois.sum((1, 2))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/photon_counting_ct/plot.py
"""Animate: the gantry turns and the material maps form, then a virtual monoenergetic image sweeps the energy."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import NullLocator

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import tensors

SPIN = 32  # logical steps of phase 1 (gantry rotation)


def animate(
    x: torch.Tensor,  # (N,)
    theta: torch.Tensor,  # (A,)
    e: torch.Tensor,  # (E,) keV
    detected: torch.Tensor,  # (B, E) open-beam spectrum per bin
    mu: torch.Tensor,  # (E, M)
    Q: torch.Tensor,  # (4, A, N, N) per-angle filtered back-projections: water, bone, iodine, conventional
    images: torch.Tensor,  # (4, N, N) final images
    maps: torch.Tensor,  # (3, N, N) true basis maps
    vmi_e: torch.Tensor,  # (K,) keV
    vmi: torch.Tensor,  # (K, N, N) HU
    vmi_rois: torch.Tensor,  # (K, 5) HU in each insert
    labels: list[str],
    doses: list[float],
    errors: list[float],
    path: Path,
) -> Path:
    n_angles = len(theta)
    spin_idx = [round(k * (n_angles - 1) / (SPIN - 1)) for k in range(SPIN)]
    running = torch.cumsum(Q, dim=1)[:, spin_idx] * (np.pi / n_angles)  # (4, SPIN, N, N) partial FBP sums
    running = running.float().cpu().numpy()
    images, vmi, vmi_rois = images.float().cpu().numpy(), vmi.float().cpu().numpy(), vmi_rois.cpu().numpy()
    e, detected, vmi_e = e.cpu().numpy(), detected.cpu().numpy(), vmi_e.cpu().numpy()
    mu_iodine = mu[:, 2].cpu().numpy() * 10.0  # 10 mg/mL iodine, 1/cm
    theta = theta.cpu().numpy()
    i70 = int(np.argmin(np.abs(vmi_e - 70)))
    mu70 = tensors.basis_attenuation(torch.tensor([vmi_e[i70]], dtype=torch.float64))[0].numpy()
    n_vmi = len(vmi_e)
    r = tensors.FOV_HALF_WIDTH
    ext = (-r, r, -r, r)

    fig, axes = plt.subplots(2, 4, figsize=(11.5, 5.4))
    (ax_sp, ax_cv, ax_io, ax_bo), (ax_wa, ax_vmi, ax_hu, ax_dose) = axes

    # spectrum seen by each energy bin and the iodine K-edge
    for b in range(detected.shape[0]):
        ax_sp.fill_between(e, detected[b] / detected.max(), color=ACCENT[b], alpha=0.5, lw=0, label=f"bin {b + 1}")
    ax_sp.set_xlabel("photon energy (keV)")
    ax_sp.set_ylabel("counts (open beam)")
    ax_sp.set_title("energy bins")
    ax_sp.plot(e, mu_iodine / mu_iodine.max(), color="#374151", lw=1.2, ls="--", label=r"iodine $\mu$ (scaled)")
    ax_sp.legend(fontsize=7, loc="upper right")

    def image(ax, data, cmap, vmin, vmax, title, unit):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        im = ax.imshow(data, cmap=plt.get_cmap(cmap, 24), vmin=vmin, vmax=vmax, origin="lower", extent=ext)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label=unit)
        ax.set_title(title, fontsize=10)
        return im

    im_cv = image(ax_cv, images[3], "gray", 0.17, 0.30, "energy-integrating CT", r"$\mu_{\rm eff}$ (1/cm)")
    im_io = image(ax_io, images[2], "magma", -2, 16, "iodine map", "mg/mL")
    im_bo = image(ax_bo, 100 * images[1], "bone", -5, 25, "bone map", "% bone")
    im_wa = image(ax_wa, images[0], "Blues_r", 0.0, 1.2, "water map", "fraction")
    im_vmi = image(ax_vmi, vmi[i70], "gray", -150, 900, "", "HU")
    for k, lab in enumerate(labels):  # name the inserts on the conventional image
        phi = np.pi / 2 + 2 * np.pi * k / 5
        ax_cv.text(9.4 * np.cos(phi), 9.4 * np.sin(phi), lab, fontsize=6.5, ha="center", va="center", color=ACCENT[1])
    (gantry,) = ax_cv.plot([], [], color=ACCENT[0], lw=3)

    for k in (0, 1, 2, 3, 4):
        style = dict(color="#374151", lw=2) if k == 4 else dict(color=ACCENT[k], lw=1.3)
        ax_hu.plot(vmi_e, vmi_rois[:, k], label=labels[k], **style)
    marker = ax_hu.axvline(vmi_e[i70], color="#f59e0b", lw=1.5)
    ax_hu.set_xlabel("virtual monoenergetic energy (keV)")
    ax_hu.set_ylabel("insert CT number (HU)")
    ax_hu.set_title("inserts in the VMI")
    ax_hu.legend(fontsize=7, loc="upper right")

    ax_dose.loglog(doses, errors, "o-", color=ACCENT[0], ms=4, label="measured")
    ref = errors[0] * np.sqrt(doses[0] / np.asarray(doses))
    ax_dose.loglog(doses, ref, "--", color="#9ca3af", lw=1, label=r"$\propto 1/\sqrt{N_0}$")
    ax_dose.set_xlabel(r"photons per ray $N_0$")
    ax_dose.set_ylabel("iodine RMS error (mg/mL)")
    ax_dose.set_title("quantification vs dose")
    ax_dose.legend(fontsize=7)
    ax_dose.xaxis.set_minor_locator(NullLocator())
    ax_dose.yaxis.set_minor_locator(NullLocator())
    ticks = [t for t in (0.1, 0.25, 0.5, 1, 2, 4, 8, 16) if min(errors) / 1.5 <= t <= max(errors) * 1.5]
    ax_dose.set_yticks(ticks, [f"{t:g}" for t in ticks])

    def update(step: int):
        if step < SPIN:  # phase 1: the gantry turns, the partial back-projections fill in
            frame = running[:, step]
            im_cv.set_data(frame[3])
            im_io.set_data(frame[2])
            im_bo.set_data(100 * frame[1])
            im_wa.set_data(frame[0])
            v = 1000 * (np.einsum("m,myx->yx", mu70, frame[:3]) - mu70[0]) / mu70[0]
            im_vmi.set_data(v)
            k_vmi = i70
            th = theta[spin_idx[step]]
            d, ray = np.array([np.cos(th), np.sin(th)]), np.array([-np.sin(th), np.cos(th)])
            end = -11.5 * ray
            gantry.set_data([end[0] - 9 * d[0], end[0] + 9 * d[0]], [end[1] - 9 * d[1], end[1] + 9 * d[1]])
            ax_cv.set_title(f"energy-integrating CT ({spin_idx[step] + 1}/{n_angles})", fontsize=10)
        else:  # phase 2: sweep the virtual monoenergetic energy
            for im, data in ((im_cv, images[3]), (im_io, images[2]), (im_bo, 100 * images[1]), (im_wa, images[0])):
                im.set_data(data)
            gantry.set_data([], [])
            ax_cv.set_title("energy-integrating CT", fontsize=10)
            k_vmi = min(n_vmi - 1, step - SPIN)
            im_vmi.set_data(vmi[k_vmi])
        marker.set_xdata([vmi_e[k_vmi], vmi_e[k_vmi]])
        ax_vmi.set_title(f"virtual mono {vmi_e[k_vmi]:.0f} keV", fontsize=10)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(SPIN + n_vmi, 60), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'kvp': 120,
    'dose': 1000000.0,
    'thresholds': '25,35,50,70',
    'sigma_kev': 3.0,
    'n_pixels': 128,
    'n_angles': 180
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.photon_counting_ct")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.photon_counting_ct.main")
problem_dir = Path("problems/medical/photon_counting_ct")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
